# YOLO Mask Overlay (Colab)

1. Upload a YOLO `.pt` model — the available classes are printed.
2. Upload a video.
3. Type the classes you want masked. Optionally alias multiple classes to a single label/color (e.g. `rough trail` + `rocky trail` → `trail`).
4. Render an annotated MP4 with masks + class names, then download.

In [ ]:
!pip install -q ultralytics opencv-python

In [ ]:
import tempfile
from pathlib import Path

import cv2
import numpy as np
from google.colab import files
from ultralytics import YOLO

WORK_DIR = Path(tempfile.mkdtemp(prefix="yolo_overlay_"))
STATE = {"model": None, "video_path": None, "names": {}}
print(f"Working directory: {WORK_DIR}")

## 1. Upload model and list classes

In [ ]:
uploaded = files.upload()  # pick the .pt
name = next(iter(uploaded))
model_path = WORK_DIR / name
model_path.write_bytes(uploaded[name])
STATE["model"] = YOLO(str(model_path))
STATE["names"] = STATE["model"].names
print(f"Loaded {name}\n\nAvailable classes:")
for idx, cname in STATE["names"].items():
    print(f"  {idx:>3}: {cname}")
task = getattr(STATE["model"], "task", None)
if task and task != "segment":
    print(f"\nWarning: model task is '{task}'. Pixel masks require a segmentation model. Falling back to filled bounding boxes.")

## 2. Upload video

In [ ]:
uploaded = files.upload()  # pick the video
name = next(iter(uploaded))
vp = WORK_DIR / name
vp.write_bytes(uploaded[name])
STATE["video_path"] = str(vp)

cap = cv2.VideoCapture(str(vp))
n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"Loaded {name}: {w}x{h}, {fps:.2f} fps, {n} frames")

## 3. Pick classes and aliases

- `KEEP`: comma-separated class names or ids (case-insensitive).
- `ALIASES`: optional. Maps any original class name to a display label. Classes that share a display label share one color and one mask layer.

In [ ]:
KEEP = "rough trail, rocky trail"

ALIASES = {
    "rough trail": "trail",
    "rocky trail": "trail",
}

_name_to_id = {n.lower(): i for i, n in STATE["names"].items()}
selected_ids = []
for token in KEEP.split(","):
    t = token.strip().lower()
    if not t:
        continue
    if t.isdigit() and int(t) in STATE["names"]:
        selected_ids.append(int(t))
    elif t in _name_to_id:
        selected_ids.append(_name_to_id[t])
    else:
        raise ValueError(f"Unknown class: {token!r}")
selected_ids = sorted(set(selected_ids))

_aliases_lc = {k.lower(): v for k, v in ALIASES.items()}
def display_label(class_id):
    raw = STATE["names"][class_id]
    return _aliases_lc.get(raw.lower(), raw)

print("Keeping:")
for i in selected_ids:
    raw = STATE["names"][i]
    lbl = display_label(i)
    suffix = f"  ->  {lbl}" if lbl != raw else ""
    print(f"  {i:>3}: {raw}{suffix}")

## 4. Render annotated video

In [ ]:
CONF = 0.25
ALPHA = 0.5

def color_for(label):
    seed = sum(ord(ch) for ch in label) * 9973 + 7
    rng = np.random.default_rng(seed)
    return tuple(int(c) for c in rng.integers(64, 255, size=3))

if STATE["model"] is None or STATE["video_path"] is None:
    raise RuntimeError("Upload model and video first.")
if not selected_ids:
    raise RuntimeError("Pick at least one class in KEEP.")

model = STATE["model"]
video_path = STATE["video_path"]
is_seg = getattr(model, "task", None) == "segment"

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
TOTAL = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

out_path = WORK_DIR / "output_overlay.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(str(out_path), fourcc, fps, (W, H))

frame_idx = 0
for result in model.predict(source=video_path, stream=True, conf=CONF, classes=selected_ids, verbose=False):
    frame = result.orig_img.copy()
    boxes = result.boxes
    masks = result.masks if is_seg else None

    label_masks = {}  # display_label -> binary mask (H, W)

    if boxes is not None and len(boxes) > 0:
        cls = boxes.cls.cpu().numpy().astype(int)
        xyxy = boxes.xyxy.cpu().numpy().astype(int)
        mask_data = masks.data.cpu().numpy() if masks is not None else None

        for i in range(len(cls)):
            c = int(cls[i])
            if c not in selected_ids:
                continue
            lbl = display_label(c)

            if mask_data is not None:
                m = mask_data[i]
                if m.shape != (H, W):
                    m = cv2.resize(m, (W, H), interpolation=cv2.INTER_LINEAR)
                m = (m >= 0.5).astype(np.uint8)
            else:
                x1, y1, x2, y2 = xyxy[i]
                m = np.zeros((H, W), dtype=np.uint8)
                m[max(0, y1):max(0, y2), max(0, x1):max(0, x2)] = 1

            if lbl in label_masks:
                label_masks[lbl] = np.maximum(label_masks[lbl], m)
            else:
                label_masks[lbl] = m

    if label_masks:
        overlay = frame.astype(np.float32).copy()
        any_mask = np.zeros((H, W), dtype=np.float32)
        color_layer = np.zeros_like(overlay)
        weight = np.zeros((H, W), dtype=np.float32)

        for lbl, m in label_masks.items():
            color = np.array(color_for(lbl), dtype=np.float32)
            mf = m.astype(np.float32)
            color_layer += mf[..., None] * color
            weight += mf
            any_mask = np.maximum(any_mask, mf)

        w_safe = np.clip(weight, 1.0, None)[..., None]
        color_layer = color_layer / w_safe
        a = (ALPHA * any_mask)[..., None]
        frame = np.clip(overlay * (1 - a) + color_layer * a, 0, 255).astype(np.uint8)

        for lbl, m in label_masks.items():
            num, _, stats, centroids = cv2.connectedComponentsWithStats(m, connectivity=8)
            color = color_for(lbl)
            for k in range(1, num):
                if stats[k, cv2.CC_STAT_AREA] < 200:
                    continue
                cx, cy = centroids[k]
                cx, cy = int(cx), int(cy)
                font = cv2.FONT_HERSHEY_SIMPLEX
                scale = 0.6
                thickness = 2
                (tw, th), baseline = cv2.getTextSize(lbl, font, scale, thickness)
                x = max(0, min(W - tw - 4, cx - tw // 2))
                y = max(th + 4, min(H - 4, cy + th // 2))
                cv2.rectangle(frame, (x - 2, y - th - 4), (x + tw + 2, y + baseline), (0, 0, 0), -1)
                cv2.putText(frame, lbl, (x, y), font, scale, color, thickness, cv2.LINE_AA)

    writer.write(frame)
    frame_idx += 1
    if frame_idx % 20 == 0 or frame_idx == TOTAL:
        print(f"frame {frame_idx}/{TOTAL}")

writer.release()
print(f"\nWrote {out_path}")

## 5. Download

In [ ]:
files.download(str(out_path))